Load the data and do an inspection first

In [1]:
%pip install pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd

df = pd.read_csv("../data/raw_data.csv")

print(df.shape)
df.info()
df.head(3)

#clean per-column count of missing values
df.isnull().sum()

(418, 8)
<class 'pandas.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   title        418 non-null    str  
 1   company      418 non-null    str  
 2   location     366 non-null    str  
 3   tags         418 non-null    str  
 4   description  418 non-null    str  
 5   apply_url    418 non-null    str  
 6   date_posted  418 non-null    str  
 7   search_tag   418 non-null    str  
dtypes: str(8)
memory usage: 26.3 KB


title           0
company         0
location       52
tags            0
description     0
apply_url       0
date_posted     0
search_tag      0
dtype: int64

In [3]:
df["company"] = df["company"].fillna("Not specified")
df["location"] = df["location"].fillna("Not specified")
df["tags"] = df["tags"].fillna("")
df["description"] = df["description"].fillna("")

In [4]:
before = len(df)
df = df.drop_duplicates(subset=["title", "company", "apply_url"])
after = len(df)
print(f"Removed {before - after} duplicate rows")

Removed 0 duplicate rows


In [5]:
text_cols = ["title", "company", "location", "tags", "description"]

for col in text_cols:
    df[col] = df[col].astype(str).str.lower().str.strip()

In [6]:
import re

def clean_text_artifacts(text):
    text = re.sub(r"\s+", " ", text)      # collapse multiple spaces/newlines into one
    text = text.replace("\xa0", " ")       # remove non-breaking space characters
    text = text.strip()
    return text

for col in text_cols:
    df[col] = df[col].apply(clean_text_artifacts)

In [8]:
df[["title", "company", "location", "description"]].sample(5)

,title,company,location,description
16,data analyst excel,yo it consulting,"sydney, sydney, new south wales, australia",job description job title: data analyst excel ...
15,online consumer research panelist data entry c...,apexfocusgroup llc,"nowra,",apex focus group partners with research organi...
129,pcv bus driver,arriva group,"hinckley,",start your career with arriva as a pcv bus dri...
200,data analyst ii,computercare,remote,computercare has spent more than 20 years buil...
54,social content designer & producer,josie maran,"remote,",position: social content designer & producerco...


In [9]:
print((df["description"].str.len() < 20).sum(), "postings have very short descriptions")

0 postings have very short descriptions


In [10]:
df.to_csv("../data/cleaned_data.csv", index=False)
print(f"Final cleaned dataset: {len(df)} rows")

Final cleaned dataset: 418 rows
